In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

print("KoGPT2 원본 모델 로드 중...")
model_name = "skt/kogpt2-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()
model.to("cuda")  # Colab에서만

def chat_kogpt2(prompt, max_length=60, top_p=0.92, temperature=0.7):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        input_ids,
        max_length=input_ids.shape[1]+30,  # 30 token만 생성
        do_sample=True,
        top_p=top_p,
        temperature=temperature,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
    )
    result = tokenizer.decode(output[0], skip_special_tokens=True)
    answer = result[len(prompt):].strip()
    # "사용자:" 또는 "챗봇:"이 다시 나오면 앞부분만
    for stopword in ["사용자:", "챗봇:"]:
        if stopword in answer:
            answer = answer.split(stopword)[0].strip()
    answer = answer.split('\n')[0].strip()
    return answer



In [ ]:
import torch

In [ ]:
def chat_kogpt2(prompt, max_length=60, top_p=0.92, temperature=0.7):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    attention_mask = torch.ones_like(input_ids)
    output = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_length=input_ids.shape[1]+30,
        do_sample=True,
        top_p=top_p,
        temperature=temperature,
        pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
    )
    result = tokenizer.decode(output[0], skip_special_tokens=True)
    answer = result[len(prompt):].strip()
    for stopword in ["사용자:", "챗봇:"]:
        if stopword in answer:
            answer = answer.split(stopword)[0].strip()
    answer = answer.split('\n')[0].strip()
    return answer

In [ ]:
print("KoGPT2 원본 챗봇 테스트 모드 시작!")

history = ""
while True:
    user = input("나: ")
    if user.strip().lower() in ["exit", "quit", "종료"]:
        print("대화를 종료합니다.")
        break
    history += f"사용자: {user}\n챗봇:"
    answer = chat_kogpt2(history)
    print("챗봇:", answer)
    history += f"{answer}\n"